YieldMax AI: Autonomous Revenue Intelligence & Predictive Pricing Engine for Global Airbnb Markets

In [28]:
!pip install duckdb xgboost langchain_openai
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


High-Performance Data Ingestion

In [30]:
# Create an in-memory high-speed database
con = duckdb.connect(database=':memory:')

# Pro Step: Load and Clean in one SQL shot
# This fixes the currency symbols and missing values automatically
con.execute("""
    CREATE TABLE raw_listings AS
    SELECT
        id,
        neighbourhood_group_cleansed,
        room_type,
        accommodates,
        CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS FLOAT) AS price,
        CAST(REPLACE(COALESCE(REPLACE(host_response_rate, '%', ''), '0'), 'N/A', '0') AS INT) AS response_rate,
        availability_365,
        number_of_reviews
    FROM read_csv_auto('listings.csv.gz')
""")

print("Phase 1: Ingestion Complete. Data is cleaned and indexed.")

Phase 1: Ingestion Complete. Data is cleaned and indexed.


AI-Agentic Feature Engineering

In [31]:
# Calculate Occupancy and Revenue Potential
con.execute("""
    CREATE TABLE analytical_engine AS
    SELECT *,
        (365 - availability_365) / 365.0 AS occupancy_rate,
        (price * (365 - availability_365)) AS actual_revenue,
        -- PRO FEATURE: Find 'Underpriced' listings with high demand
        CASE WHEN (365 - availability_365) > 300 AND price < 100 THEN 1 ELSE 0 END AS revenue_leakage_flag
    FROM raw_listings
""")

df = con.table('analytical_engine').df()
df.head()

,id,neighbourhood_group_cleansed,room_type,accommodates,price,response_rate,availability_365,number_of_reviews,occupancy_rate,actual_revenue,revenue_leakage_flag
0,5456,None,Entire home/apt,3,97.0,100,328,708,0.101370,3589.0,0
1,6448,None,Entire home/apt,2,160.0,100,316,339,0.134247,7840.0,0
2,8502,None,Entire home/apt,2,38.0,100,88,54,0.758904,10526.0,0
3,13035,None,Entire home/apt,3,145.0,100,321,19,0.120548,6380.0,0
4,22828,None,Entire home/apt,2,58.0,100,211,56,0.421918,8932.0,0


Machine Learning (Predicting Future Revenue)

In [34]:
# Prepare data for XGBoost
# Drop rows where 'actual_revenue' is NaN, as XGBoost cannot handle them in the target variable
df_cleaned = df.dropna(subset=['actual_revenue'])
X = df_cleaned[['accommodates', 'response_rate', 'number_of_reviews']]
y = df_cleaned['actual_revenue']

model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
model.fit(X, y)

# Save the model predictions back to our table
df_cleaned['predicted_revenue'] = model.predict(X)
df_cleaned.to_csv('pro_yield_data.csv', index=False)
print("Phase 2: ML Engine Trained. Exported for Power BI.")

Phase 2: ML Engine Trained. Exported for Power BI.


/tmp/ipykernel_4139/1325630224.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['predicted_revenue'] = model.predict(X)
